In [ ]:
# ============================================================
# GOLD PRICE FORECASTING – DATA PREPROCESSING PIPELINE
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# PHASE 1 — LOAD DATA
# ------------------------------------------------------------

gold = pd.read_csv("XAU_USD Historical Data.csv")
google = pd.read_csv("google_trends_gold_keywords_daily_2008_2017.csv")
sentiment = pd.read_csv("gold-sentiment-dataset-cleaned.csv")
vix = pd.read_csv("CBOE Volatility Index Historical Data.csv")
usd = pd.read_csv("US Dollar Index Futures Historical Data.csv")

# ------------------------------------------------------------
# PHASE 2 — CLEAN & STANDARDISE
# ------------------------------------------------------------

def clean_price_data(df, date_col="Date"):
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.sort_values(date_col)
    if "Vol." in df.columns:
            df = df.drop(columns=["Vol."])
    # Convert numeric columns that may contain commas or %
    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = (
                df[col]
                .str.replace(",", "", regex=False)
                .str.replace("%", "", regex=False)
            )
            try:
                df[col] = df[col].astype(float)
            except:
                pass
    return df


    return df

gold = clean_price_data(gold)
vix = clean_price_data(vix)
usd = clean_price_data(usd)

google["date"] = pd.to_datetime(google["date"])
google = google.sort_values("date")

# Sentiment date format issues were already fixed in the ""gold_sentiment_dataset clean.ipynb"" notebook
sentiment["date"] = pd.to_datetime(sentiment["date"])
sentiment = sentiment.sort_values("date")
### Filter dataframe to relevant period (2008-2017)
# Define start and end dates
start_date = "2008-01-01"
end_date   = "2017-12-31"

# Filter rows between the two dates
sentiment = sentiment[(sentiment['date'] >= start_date) & (sentiment['date'] <= end_date)]

# ------------------------------------------------------------
# PHASE 3 — ALIGN TO GOLD TRADING DAYS
# ------------------------------------------------------------

# Use gold trading calendar as master
data = gold.copy()

# Merge VIX and USD
data = data.merge(vix[["Date", "Price"]], on="Date", how="left", suffixes=("", "_vix"))
data = data.merge(usd[["Date", "Price"]], on="Date", how="left", suffixes=("", "_usd"))

data.rename(columns={
    "Price": "Gold_Close",
    "Price_vix": "VIX_Close",
    "Price_usd": "USD_Close"
}, inplace=True)

# Merge Google Trends (forward fill weekends)
google = google.set_index("date").reindex(data["Date"]).ffill().reset_index()
google.rename(columns={"index": "Date"}, inplace=True)

data = data.merge(google, on="Date", how="left")

# ------------------------------------------------------------
# PHASE 4 — AGGREGATE SENTIMENT TO DAILY LEVEL
# ------------------------------------------------------------

sent_daily = sentiment.groupby("date").agg({
    "Price Direction Up": "sum",
    "Price Direction Down": "sum",
    "Future Information": "sum",
    "News": "count"
}).reset_index()

sent_daily.rename(columns={
    "date": "Date",
    "News": "Headline_Count"
}, inplace=True)

# Net sentiment
sent_daily["Net_Sentiment"] = (
    sent_daily["Price Direction Up"] -
    sent_daily["Price Direction Down"]
)

# Merge
data = data.merge(sent_daily, on="Date", how="left")

# Fill only sentiment-related columns with 0
sentiment_cols = [
    "Price Direction Up",
    "Price Direction Down",
    "Future Information",
    "Headline_Count",
    "Net_Sentiment"
]

for col in sentiment_cols:
    if col in data.columns:
        data[col] = data[col].fillna(0)


# ------------------------------------------------------------
# PHASE 5 — FEATURE ENGINEERING
# ------------------------------------------------------------

# Gold price range
data["Daily_Range"] = data["High"] - data["Low"]

# Rolling features
for window in [5, 10]:
    data[f"Gold_MA_{window}"] = data["Gold_Close"].rolling(window).mean()
    data[f"Gold_STD_{window}"] = data["Gold_Close"].rolling(window).std()
    data[f"VIX_MA_{window}"] = data["VIX_Close"].rolling(window).mean()
    data[f"USD_MA_{window}"] = data["USD_Close"].rolling(window).mean()

# Google rolling means
google_cols = ["gold price", "gold forecast", "buy gold", "inflation", "safe haven"]

for col in google_cols:
    data[f"{col}_MA_5"] = data[col].rolling(5).mean()

# ------------------------------------------------------------
# PHASE 6 — LAG FEATURES (PREVENT DATA LEAKAGE)
# ------------------------------------------------------------

lag_cols = [
    "VIX_Close", "USD_Close",
    "Daily_Range", "Net_Sentiment", "Headline_Count"
] + google_cols

for col in lag_cols:
    data[f"{col}_lag1"] = data[col].shift(1)

# Drop original columns that shouldn't be used directly
data = data.drop(columns=lag_cols)

# ------------------------------------------------------------
# PHASE 7 — CREATE MULTI-HORIZON TARGETS (1–10 DAYS)
# ------------------------------------------------------------

HORIZON = 10

for h in range(1, HORIZON + 1):
    data[f"Target_t+{h}"] = data["Gold_Close"].shift(-h)

# ------------------------------------------------------------
# PHASE 8 — REMOVE ROWS WITH NA (FROM ROLLING + SHIFTING)
# ------------------------------------------------------------

data = data.dropna().reset_index(drop=True)

# ------------------------------------------------------------
# PHASE 9 — CREATE BASE VS ALTERNATIVE DATASETS
# ------------------------------------------------------------

# Base dataset (no Google, no Sentiment)
base_features = [col for col in data.columns
                 if not any(x in col for x in google_cols)
                 and "Sentiment" not in col
                 and "Headline" not in col
                 and "Target" not in col
                 and col != "Date"]

alt_features = [col for col in data.columns
                if col not in ["Date"] 
                and "Target" not in col]

# Create separate datasets
base_dataset = data[["Date"] + base_features +
                    [f"Target_t+{h}" for h in range(1, HORIZON+1)]]

alt_dataset = data[["Date"] + alt_features +
                   [f"Target_t+{h}" for h in range(1, HORIZON+1)]]

# ------------------------------------------------------------
# SAVE CLEAN DATASETS
# ------------------------------------------------------------

base_dataset.to_csv("processed_base_dataset.csv", index=False)
alt_dataset.to_csv("processed_alt_dataset.csv", index=False)

print("Preprocessing complete.")
print("Base dataset shape:", base_dataset.shape)
print("Alt dataset shape:", alt_dataset.shape)


Preprocessing complete.
Base dataset shape: (1761, 30)
Alt dataset shape: (1761, 42)


In [12]:
print(data.columns)

Index(['Date', 'Open', 'High', 'Low', 'Vol.', 'Change %', 'Price Direction Up',
       'Price Direction Down', 'Future Information', 'Gold_MA_5', 'Gold_STD_5',
       'VIX_MA_5', 'USD_MA_5', 'Gold_MA_10', 'Gold_STD_10', 'VIX_MA_10',
       'USD_MA_10', 'gold price_MA_5', 'gold forecast_MA_5', 'buy gold_MA_5',
       'inflation_MA_5', 'safe haven_MA_5', 'Gold_Close_lag1',
       'VIX_Close_lag1', 'USD_Close_lag1', 'Daily_Range_lag1',
       'Net_Sentiment_lag1', 'Headline_Count_lag1', 'gold price_lag1',
       'gold forecast_lag1', 'buy gold_lag1', 'inflation_lag1',
       'safe haven_lag1'],
      dtype='object')
